# Eligibility-aware RAG credit-card recommender: final end-to-end notebook
**Runtime:** Colab T4 GPU. **Needs:** `Thesis-Coding-final.zip` in `MyDrive/Thesis/`.

Run top to bottom. Steps 1-9 take about 15 minutes and validate everything (integrity checks, 33 unit/end-to-end tests, real-model sanity check). Steps 11-14 are the long evaluation (several hours on a T4). **Every generation is cached on Drive and every result is saved to Drive**, so after a disconnect re-run Steps 1-5 and 8, then simply re-run the evaluation cells: finished work is replayed instantly.

## Step 1: Mount Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)} | free {free/1024**3:.1f} / {total/1024**3:.1f} GiB")

## Step 2: Settings (run before any project import)

In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/Thesis/Thesis-Coding-final.zip"
RESULTS_DIR = "/content/drive/MyDrive/Thesis/results_final"        # all numbers + predictions + RESULTS.md
os.makedirs(RESULTS_DIR, exist_ok=True)

os.environ["USE_STUB_LLM"] = "false"
os.environ["LLM_MODEL_NAME"] = "Qwen/Qwen2.5-7B-Instruct"             # 4-bit; fallback: "Qwen/Qwen2.5-3B-Instruct"
os.environ["LLM_CACHE_PATH"] = "/content/drive/MyDrive/Thesis/llm_cache.jsonl"   # survives disconnects
os.environ["LLM_CACHE"] = "1"
os.environ["LLM_BATCH_SIZE"] = "8"                                    # per-card prompts are generated 8 at a time
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # optional

## Step 3: Unpack the project

In [ ]:
!rm -rf /content/Thesis-Coding && mkdir -p /content/Thesis-Coding
!unzip -q -o "$ZIP_PATH" -d /content/Thesis-Coding
%cd /content/Thesis-Coding
assert "round 4" in open("CHANGES.md").read().lower(), "Old zip: upload Thesis-Coding-final.zip"
!ls

## Step 4: Install dependencies

In [ ]:
%pip install -q -r requirements.txt "requests==2.32.4"

## Step 5: Build the FAISS index and chunk table

In [ ]:
!python -m src.pipeline
assert os.path.exists("artifacts/faiss.index") and os.path.exists("artifacts/chunks.parquet")

## Step 6: Integrity checks and tests
`verify_setup` confirms the rule documents agree with the eligibility table, the gold labels equal the oracle, the index matches the documents, and the grounded/ungrounded prompts are built correctly. **Both must pass before any result is trusted.** Expect `33 passed`.

In [ ]:
!python -m evaluation.verify_setup
!python -m pytest -q tests 2>&1 | tail -3

## Step 7: Plumbing check with the stub LLM (about 1 minute)
Stub numbers only test the code path; they are not model results.

In [ ]:
!python -m evaluation.run_ablation --use-stub-llm --split test --limit 6 --out-dir /content/stub_results --arms A2 A3 A5 A6 2>&1 | grep -E "===|Results saved|Error|Traceback" 

## Step 8: Load the real model and sanity-test single and batched generation

In [ ]:
from src.infrastructure.llm_client import LocalLLMClient

client = LocalLLMClient(os.environ["LLM_MODEL_NAME"])
print("model:", client.model_name, "| max_new_tokens:", client.max_new_tokens)
print(client.generate("Reply with exactly: GPU test passed."))
print(client.generate_batch(["Reply with exactly: A", "Reply with exactly: B"]))

## Step 9: Real-pipeline check on 3 profiles (grounded and ungrounded)
**Pass:** grounded `STRUCTURE OK` (8 parsed, nothing missing/extra) **and** the ungrounded arm also returns all 8 cards (otherwise its errors would be output-format failures, not wrong knowledge). `WRONG` is model accuracy.

In [ ]:
from pathlib import Path
from evaluation.run_ablation import load_gold_set, build_retriever, make_llm_clients
from src.model import RAGRecommender

rows = Path("data/gold_test_set.jsonl").read_text(encoding="utf-8").splitlines()
Path("/content/gold_smoke.jsonl").write_text("\n".join(rows[:3]) + "\n", encoding="utf-8")
records = load_gold_set("/content/gold_smoke.jsonl")

grounded_llm, ungrounded_llm = make_llm_clients(use_stub=False)
retriever = build_retriever(4)
grounded = RAGRecommender(retriever, grounded_llm)
ungrounded = RAGRecommender(retriever, ungrounded_llm)

structural_ok = True
for i, rec in enumerate(records):
    expected = {(d.bank, d.card): d.recommended for d in rec.expected}
    for name, system in (("GROUNDED  ", grounded), ("UNGROUNDED", ungrounded)):
        decisions, k = system.recommend(rec.profile)
        actual = {(d.bank, d.card): d.recommended for d in decisions}
        missing, extra = set(expected) - set(actual), set(actual) - set(expected)
        wrong = {c[1]: (expected[c], actual[c]) for c in expected.keys() & actual.keys() if expected[c] != actual[c]}
        if name.strip() == "GROUNDED":
            structural_ok &= (not missing and not extra and len(decisions) == len(expected))
        print(f"Profile {i} {name}: parsed={len(decisions)}/{len(expected)} MISSING={len(missing)} WRONG={wrong or '-'}")
    d0 = grounded.recommend(rec.profile)[0][0]
    print("   e.g.", d0.card, d0.recommended, "|", d0.reason[:150], "|", d0.cited_chunk_id)
assert structural_ok, "Grounded output structure problem: paste this output"
print("\nSTRUCTURE OK")

## Step 9b: Free GPU/RAM before the evaluation subprocesses

In [ ]:
import gc, torch
for n in ("client", "grounded_llm", "ungrounded_llm", "grounded", "ungrounded", "retriever"):
    globals().pop(n, None)
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.mem_get_info()[0]/1024**3:.1f} GiB free")

## Step 10 (optional): quick pilot, 10 profiles
Gives a first real number and the seconds per profile for planning. Skip if you are short on time.

In [ ]:
!python -m evaluation.run_ablation --split test --limit 10 --out-dir /content/pilot --arms 2>&1 | tr '\r' '\n' | grep -v "it/s" | grep -E "===|accuracy|RQ|verdict|returned" 

## Step 11: MAIN RESULT: test split (40 profiles), A1 + RQ1 / RQ2 / RQ3 / RQ4
The test split is every 3rd gold profile. Results are merged into `results.json` on Drive.

In [ ]:
!python -m evaluation.run_ablation --split test --out-dir $RESULTS_DIR --arms 2>&1 | tr '\r' '\n' | grep --line-buffered -v "it/s" | tee $RESULTS_DIR/main_test40.log

## Step 12: Ablation arms (20 test profiles each; `--no-core` keeps the main result intact)
One cell per arm; run them in any order.

In [ ]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A3 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|prompt_pattern" | tee $RESULTS_DIR/a3.log

In [ ]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A5 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|accuracy" | tee $RESULTS_DIR/a5.log

In [ ]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A2 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|top_k" | tee $RESULTS_DIR/a2.log

In [ ]:
!python -m evaluation.run_ablation --split test --limit 40 --no-core --out-dir $RESULTS_DIR --arms A6 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|balanced|normal_|reworded_" | tee $RESULTS_DIR/a6.log

## Step 13 (optional): A4 model selection (Qwen2.5-3B vs 7B)

In [ ]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A4 --a4-models Qwen/Qwen2.5-3B-Instruct Qwen/Qwen2.5-7B-Instruct 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|Qwen" | tee $RESULTS_DIR/a4.log

## Step 14 (optional but recommended): held-out confirmation on the dev split
The 40 test profiles were looked at during earlier development rounds. Running the final configuration on 40 dev profiles (never evaluated before) gives a clean estimate to report next to the main result.

In [ ]:
!python -m evaluation.run_ablation --split dev --limit 40 --out-dir $RESULTS_DIR/dev_confirmation --arms 2>&1 | tr '\r' '\n' | grep --line-buffered -v "it/s" | tee $RESULTS_DIR/dev_confirmation.log

## Step 15: Build the shareable report (`RESULTS.md`, saved on Drive)

In [ ]:
!python -m evaluation.make_report --results-dir $RESULTS_DIR

In [ ]:
# dev confirmation report (if you ran Step 14)
import os
if os.path.exists(f"{RESULTS_DIR}/dev_confirmation/results.json"):
    get_ipython().system(f"python -m evaluation.make_report --results-dir {RESULTS_DIR}/dev_confirmation")

## What to send your supervisor
`MyDrive/Thesis/results_final/`: `RESULTS.md` (all tables and error analysis), `results.json` (every number with model, split and timings), `predictions_A1_*.jsonl` (per-profile expected vs actual with the model's reasons), the `*.log` files, and the project zip with `EVALUATION_NOTES.md` and `CHANGES.md`.

## Step 16 (optional): serve the API
Run in its own session: `!uvicorn src.api:app --host 0.0.0.0 --port 8000`. With the real LLM each request issues 8 batched per-card generations.